In [ ]:
# ===================================================================================================
# SSC-MEK PHASE B3. The declared mesenchymal-only (ASC) secondary.
#
# Pre-registration 45de35e2..., Section 8 item 1: "the same analysis restricted to ASC signatures
# (1,776 compounds, one signature each)... reported and never gating." Section 8 also states WHY it
# is secondary: it rests on a single signature per identifier, and a single signature scores exactly
# zero far more often than a median over many.
#
# STATED BEFORE THE RUN, so that it cannot be read back as an excuse afterward:
#   Phase B2 found exact-zero connectivity for 45.6% of the library in Q1 and 51.2% in Q2, and both
#   PD-0325901 identifiers landed on the Q2 tied block, which made Q2 carry no compound-specific
#   information at all. With ONE signature per compound instead of a median over many, the ASC
#   exact-zero fraction is expected to be HIGHER, not lower. If it is, this secondary has even less
#   resolution than the primary, and its failure to separate PD-0325901 is a statement about the
#   instrument's resolution and not about MEK inhibition. The run reports the fraction either way.
#
# This notebook computes NO new primary and changes NO pre-registered quantity. It is bound to the
# B2 result by hash, and it reproduces B2's per-identifier percentiles before it restricts anything.
# ===================================================================================================
# CELL 1: load the frozen inputs, the library, and the B2 result. Nothing below may change them.
from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, time, subprocess, sys, collections
import numpy as np, pandas as pd
BENCH = "/content/drive/MyDrive/RitschelResearch/benchmark"
W = "/content/drive/MyDrive/RitschelResearch/ssc_mek"; os.chdir(W)
PREREG = "45de35e26156bf1c70d22bc9100adaeec30f3fc4e70b803e075c124960c3a4c9"
FREEZE = "c083e3cfb23cca67b26e772d8435b138d982db02bf74347ad13053c6c42e8301"
B2SHA  = "ed250b767fe831f4c78b9c06b4a09e114669ea6db71fd40b8a397f318cb4c6a8"
REPORT = []
def R(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); REPORT.append(s)

raw = open("sscmek_signatures_frozen_B1.json", "rb").read()
assert hashlib.sha256(raw).hexdigest() == FREEZE, ("NOT THE CHECKED B1 FREEZE", hashlib.sha256(raw).hexdigest())
FZ = json.loads(raw); SIG = FZ["signatures"]; DR = pd.DataFrame(FZ["drugs"])
assert FZ["prereg_sha256"] == PREREG, "FREEZE DOES NOT CARRY THE COMMITTED PRE-REGISTRATION HASH"
PRIM = FZ["primary_queries"]; SENS = FZ["sensitivity_queries"]

b2raw = open("sscmek_B2_result.json", "rb").read()
GOT = hashlib.sha256(b2raw).hexdigest()
assert GOT == B2SHA, ("THIS IS NOT THE B2 RESULT THAT WAS REPORTED - RE-RUN B2 OR FIX THE HASH", GOT)
B2 = json.loads(b2raw)
LIVE = B2["live_queries"]
assert B2["prereg_sha256"] == PREREG and B2["frozen_sha256"] == FREEZE, "B2 WAS NOT RUN ON THESE INPUTS"
QALL = sorted(set(LIVE) | set(SENS))

g = pd.read_csv("geneinfo_beta.txt", sep="\t")
LMG = g.loc[g.feature_space == "landmark"].copy(); LMG["gene_id"] = LMG.gene_id.astype(int)
assert len(LMG) == 978, ("LANDMARK TABLE IS NOT 978", len(LMG))
M = np.load(os.path.join(BENCH, "libA_matrix.npy"))
perts = np.load(os.path.join(BENCH, "libA_perts.npy"), allow_pickle=True)
genes = np.load(os.path.join(BENCH, "libA_genes.npy")); GI = {int(x): i for i, x in enumerate(genes)}
assert M.shape == (90834, 978) and len(perts) == 90834, M.shape

R("prereg", PREREG)
R("freeze", FREEZE)
R("B2 result", GOT, "(hash checked)")
R("library", M.shape, "compounds", len(set(perts.tolist())))
R("primary queries surviving B2's control", LIVE, "of", PRIM, "| sensitivity", SENS)
R("B2 recorded exact-zero fractions", {q: round(B2["tie_frac"][q], 3) for q in B2["tie_frac"]})
R("B2 primary: observed %.4f p %.5f" % (B2["observed"], B2["p"]))

In [ ]:
# CELL 2: scoring code. ks/conn_all are copied VERBATIM from the deposited benchmark notebook.
# A vectorized form is added for speed and is accepted only after it reproduces conn_all exactly.
def ks(rp, n, t):
    if t == 0 or n == 0: return 0.0
    V = np.sort(np.asarray(rp, float)); j = np.arange(1, t + 1, dtype=float)
    a = np.max(j / t - V / n); b = np.max(V / n - (j - 1.0) / t)
    return float(a) if a > b else float(-b)
def conn_all(Mx, up, dn):
    n = Mx.shape[1]; out = np.empty(Mx.shape[0])
    order = np.argsort(-Mx, axis=1, kind="stable")
    rank = np.empty_like(order); ar = np.arange(1, n + 1)
    np.put_along_axis(rank, order, np.broadcast_to(ar, order.shape), axis=1)
    for i in range(Mx.shape[0]):
        eu = ks(rank[i, up], n, len(up)); ed = ks(rank[i, dn], n, len(dn))
        out[i] = 0.0 if eu * ed >= 0 else (eu - ed) / 2.0
    return out
def rank_matrix(Mx, chunk=8000):
    n = Mx.shape[1]; rk = np.empty(Mx.shape, dtype=np.int32); ar = np.arange(1, n + 1, dtype=np.int32)
    for a in range(0, Mx.shape[0], chunk):
        b = min(a + chunk, Mx.shape[0])
        o = np.argsort(-Mx[a:b], axis=1, kind="stable")
        np.put_along_axis(rk[a:b], o, np.broadcast_to(ar, o.shape), axis=1)
    return rk
def _ksv(sub, n, t):
    V = np.sort(sub, axis=1).astype(np.float64); j = np.arange(1, t + 1, dtype=np.float64)
    a = np.max(j / t - V / n, axis=1); b = np.max(V / n - (j - 1.0) / t, axis=1)
    return np.where(a > b, a, -b)
def conn_from_rank(rk, up, dn):
    n = rk.shape[1]
    eu = _ksv(rk[:, up], n, len(up)); ed = _ksv(rk[:, dn], n, len(dn))
    return np.where(eu * ed >= 0, 0.0, (eu - ed) / 2.0)
def midrank(v):
    v = np.asarray(v, float); n = v.size; o = np.argsort(v, kind="stable"); r = np.empty(n); i = 0
    while i < n:
        j = i
        while j + 1 < n and v[o[j + 1]] == v[o[i]]: j += 1
        r[o[i:j + 1]] = 0.5 * (i + j) + 1.0; i = j + 1
    return r
def spear(a, b):
    ra = midrank(a) - midrank(a).mean(); rb = midrank(b) - midrank(b).mean()
    d = np.sqrt((ra * ra).sum() * (rb * rb).sum())
    return float("nan") if d == 0 else float((ra * rb).sum() / d)
# POSITIVE CONTROL for the vectorized form: it must reproduce conn_all to the bit on a real subset.
_q = SIG[PRIM[0]]; _cols = [GI[e] for e in _q["usable"]]; _pos = {e: i for i, e in enumerate(_q["usable"])}
_up = np.array([_pos[e] for e in _q["up"]]); _dn = np.array([_pos[e] for e in _q["dn"]])
_sub = M[:3000][:, _cols]
_a = conn_all(_sub, _up, _dn); _b = conn_from_rank(rank_matrix(_sub), _up, _dn)
assert np.array_equal(_a, _b), ("VECTORIZED SCORE DIFFERS FROM THE DEPOSITED conn_all", float(np.abs(_a - _b).max()))
R("scoring check: the vectorized score reproduces the deposited conn_all exactly on 3,000 signatures")
R("  (nonzero scores in that subset: %d of 3000)" % int((_a != 0).sum()))

In [ ]:
# CELL 3: recover each library row's cell line, and PROVE the recovery is in the library's row order.
# The matrix was built in Phase A by sorting the QC-pass exemplar trt_cp signatures by their column
# position in the LINCS 2020 level-5 gctx. Rebuilding that order from the same two sources must
# reproduce libA_perts.npy element for element, or the cell-line vector is not aligned to the matrix
# and every ASC row below would be the wrong row. This is the same recipe the deposited benchmark
# Phase B3 used; it is re-run here rather than trusted.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fsspec", "aiohttp", "h5py"], check=False)
import fsspec, h5py
t0 = time.time()
s = pd.read_csv(os.path.join(BENCH, "siginfo_beta.txt"), sep="\t", low_memory=False,
                usecols=["sig_id", "pert_id", "pert_type", "is_exemplar_sig", "qc_pass", "cell_iname"])
n_all = len(s)
s = s[(s.pert_type == "trt_cp") & (s.is_exemplar_sig == 1) & (s.qc_pass == 1)]
GCTX = "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/level5/level5_beta_trt_cp_n720216x12328.gctx"
h = h5py.File(fsspec.open(GCTX, "rb").open(), "r")
colid = np.array([x.decode() for x in h["0"]["META"]["COL"]["id"][:]])
pos = {x: i for i, x in enumerate(colid)}
want = sorted((pos[x], p, c) for x, p, c in zip(s.sig_id, s.pert_id, s.cell_iname) if x in pos)
cells = np.array([w[2] for w in want]); p2 = np.array([w[1] for w in want])
assert len(p2) == len(perts), ("ROW COUNT NOT REPRODUCED", len(p2), len(perts))
assert (p2 == perts).all(), ("LIBRARY ROW ORDER NOT REPRODUCED - THE CELL LINE VECTOR IS NOT ALIGNED",
                             int((p2 != perts).sum()))
R("siginfo rows %d -> QC-pass exemplar trt_cp %d | gctx columns %d | %ds"
  % (n_all, len(s), len(colid), round(time.time() - t0)))
R("library row order reproduced from the gctx column ids and siginfo: libA_perts.npy matches element for element")
ASC = cells == "ASC"
R("cell lines recovered: %d distinct | ASC signatures %d of %d rows" % (len(set(cells.tolist())), int(ASC.sum()), len(cells)))
_t = collections.Counter(cells.tolist())
R("the five largest cell lines:", ", ".join("%s %d" % (k, v) for k, v in _t.most_common(5)))
# The pre-registration's own description of the ASC subset is checked, not assumed.
_ap = perts[ASC]; _per = collections.Counter(collections.Counter(_ap.tolist()).values())
R("ASC compounds %d | signatures per ASC compound: %s"
  % (len(set(_ap.tolist())), ", ".join("%d sig -> %d compounds" % (k, v) for k, v in sorted(_per.items()))))
if len(set(_ap.tolist())) != 1776:
    R("NOTE: the pre-registration said 1,776 ASC compounds. The library gives %d. The run continues on"
      % len(set(_ap.tolist())))
    R("   the library's actual content, and the write-up must carry the corrected count.")

In [ ]:
# CELL 4: score the FULL library on all three queries, keeping the per-signature scores, and
# reproduce B2's per-identifier percentiles exactly before anything is restricted. If this does not
# reproduce B2 to 1e-12 then B3 is not scoring the same thing B2 scored and nothing below is comparable.
SCORE = {}; PCT = {}; COMP = {}; TIE = {}; ZERO = {}; NC = {}
t0 = time.time()
for qid in QALL:
    q = SIG[qid]; cols = [GI[e] for e in q["usable"]]; p_ = {e: i for i, e in enumerate(q["usable"])}
    up = np.array([p_[e] for e in q["up"]]); dn = np.array([p_[e] for e in q["dn"]])
    assert len(up) == 150 and len(dn) == 150 and not set(up.tolist()) & set(dn.tolist()), ("QUERY SETS MALFORMED", qid)
    sc = conn_from_rank(rank_matrix(M[:, cols]), up, dn)
    SCORE[qid] = sc
    comp = pd.DataFrame({"p": perts, "s": sc}).groupby("p")["s"].median()
    COMP[qid] = comp
    PCT[qid] = pd.Series((midrank(comp.values) - 0.5) / len(comp), index=comp.index)
    TIE[qid] = float((comp == 0).mean()); NC[qid] = int(len(comp))
    z = comp.index[comp.values == 0.0]
    ZERO[qid] = float(PCT[qid][z[0]]) if len(z) else None
    R("scored %s on the full library: compounds %d | exact-zero fraction %.3f | tied block at %s | %ds"
      % (qid, NC[qid], TIE[qid], "none" if ZERO[qid] is None else round(ZERO[qid], 4), round(time.time() - t0)))

worst = 0.0; checked = 0
for qid, role, drug, pid, v, on in B2["detail"]:
    if qid not in PCT: continue
    d = abs(float(PCT[qid][pid]) - float(v)); worst = max(worst, d); checked += 1
assert checked > 0, "B2 DETAIL CARRIED NOTHING TO CHECK AGAINST"
assert worst < 1e-12, ("B3 DOES NOT REPRODUCE B2'S PERCENTILES", worst)
R("B2 reproduced: %d recorded identifier percentiles, worst difference %.2e" % (checked, worst))
for qid in QALL:
    if qid in B2["tie_frac"]:
        assert abs(TIE[qid] - B2["tie_frac"][qid]) < 1e-12, ("TIE FRACTION DOES NOT MATCH B2", qid)
R("exact-zero fractions match B2 exactly on", [q for q in QALL if q in B2["tie_frac"]])
PD_IDS = sorted(DR[DR.role == "primary"].pert_id)
assert len(PD_IDS) == 2, ("PD-0325901 SHOULD HAVE TWO IDS", PD_IDS)

In [ ]:
# CELL 5: the ASC restriction. The score of one signature does not depend on which other rows are in
# the library - the rank is taken within the signature, across genes. So restricting to ASC can only
# change which compounds are in the percentile denominator, never any score. That is asserted here
# rather than assumed: the ASC compound scores must be exactly the ASC rows of the full score vector.
APCT = {}; ACOMP = {}; ATIE = {}; AZERO = {}; ANC = {}
for qid in QALL:
    sub = pd.DataFrame({"p": perts[ASC], "s": SCORE[qid][ASC]})
    comp = sub.groupby("p")["s"].median()
    # the restriction changed no score: every ASC compound's score is the median of its ASC rows only
    _chk = sub.groupby("p")["s"].apply(lambda v: float(np.median(v.values)))
    assert np.allclose(comp.values, _chk.values, atol=0, rtol=0), ("ASC MEDIAN NOT A MEDIAN OF THE SAME ROWS", qid)
    ACOMP[qid] = comp
    APCT[qid] = pd.Series((midrank(comp.values) - 0.5) / len(comp), index=comp.index)
    ATIE[qid] = float((comp == 0).mean()); ANC[qid] = int(len(comp))
    z = comp.index[comp.values == 0.0]
    AZERO[qid] = float(APCT[qid][z[0]]) if len(z) else None
    R("ASC %s: compounds %d | exact-zero fraction %.3f (full library %.3f) | tied block at %s"
      % (qid, ANC[qid], ATIE[qid], TIE[qid], "none" if AZERO[qid] is None else round(AZERO[qid], 4)))
# The expectation stated in this notebook's header, checked.
_worse = [q for q in QALL if ATIE[q] > TIE[q]]
if _worse:
    R("AS STATED BEFORE THE RUN: the ASC exact-zero fraction is HIGHER than the full library's in",
      _worse, "- a single signature per compound resolves less than a median over many.")
else:
    R("CONTRARY TO THE EXPECTATION STATED BEFORE THE RUN: the ASC exact-zero fraction is not higher")
    R("   than the full library's in any query. The stated reason for keeping this secondary does not")
    R("   hold in this data, and the write-up must say so.")
_miss = {q: [p for p in PD_IDS if p not in APCT[q].index] for q in QALL}
for q in QALL:
    if _miss[q]: R("ASC %s: PD-0325901 identifier(s) absent from the ASC subset: %s" % (q, _miss[q]))

In [ ]:
# CELL 6: the ASC secondary statistic. Same structure as the pre-registered primary - median over the
# drug's identifiers within a query, then the mean over the surviving primary queries - and the same
# null structure, 10,000 draws of two distinct ASC identifiers, the same two in every query.
# DECLARED SECONDARY. REPORTED, NEVER GATING. It cannot meet or fail the pre-registered criterion.
def asc_stat(ids, queries):
    v = [APCT[q].reindex(ids).dropna().values for q in queries]
    if any(len(x) == 0 for x in v): return None
    return float(np.mean([float(np.median(x)) for x in v]))
APER = {q: (float(np.median(APCT[q].reindex(PD_IDS).dropna().values))
            if len(APCT[q].reindex(PD_IDS).dropna()) else None) for q in QALL}
for q in QALL:
    got = [p for p in PD_IDS if p in APCT[q].index]
    R("ASC PD-0325901 in %s: %s -> %s | drug percentile %s"
      % (q, got, [round(float(APCT[q][p]), 4) for p in got],
         "no ASC signature" if APER[q] is None else round(APER[q], 4)))
ALIVE = [q for q in LIVE if APER[q] is not None]
ASTAT = float(np.mean([APER[q] for q in ALIVE])) if ALIVE else None
R("ASC secondary statistic (mean over %s): %s" % (ALIVE, "undefined" if ASTAT is None else round(ASTAT, 4)))
if ASTAT is None:
    AP = None
    R("the ASC secondary is undefined: PD-0325901 has no ASC signature in a surviving primary query")
else:
    common = sorted(set.intersection(*[set(APCT[q].index) for q in ALIVE]))
    POSN = {q: {p: i for i, p in enumerate(APCT[q].index)} for q in ALIVE}
    cidx = {q: np.array([POSN[q][p] for p in common]) for q in ALIVE}
    IDXA = {q: APCT[q].values for q in ALIVE}
    B = 10000; rng = np.random.default_rng(20261001)
    R("ASC null draws from the %d ASC compounds scored in every surviving query" % len(common))
    assert len(common) >= 100, ("TOO FEW ASC COMPOUNDS FOR A 10,000-DRAW NULL", len(common))
    NA = np.empty(B); t0 = time.time()
    for b in range(B):
        pick = rng.choice(len(common), size=2, replace=False)
        NA[b] = np.mean([np.median(IDXA[q][cidx[q][pick]]) for q in ALIVE])
    AP = (1 + int((NA <= ASTAT).sum())) / (B + 1)
    R("ASC null mean %.4f sd %.4f over %d draws (%ds)" % (NA.mean(), NA.std(), B, round(time.time() - t0)))
    R("ASC secondary p = %.5f  [SECONDARY, NOT GATING - the pre-registered criterion was settled in B2]" % AP)

In [ ]:
# CELL 7: the rest of the ASC picture - every identifier, the MEK1/2 class, the references, and S1.
ADETAIL = []
for qid in QALL:
    for r in DR.itertuples():
        if r.pert_id in APCT[qid].index:
            v = float(APCT[qid][r.pert_id])
            on = (AZERO[qid] is not None) and abs(v - AZERO[qid]) < 1e-12
            ADETAIL.append((qid, r.role, r.drug, r.pert_id, v, bool(on)))
        else:
            ADETAIL.append((qid, r.role, r.drug, r.pert_id, None, False))
R("--- ASC subset: every declared identifier (percentile; lower = stronger reversal) ---")
for qid, role, drug, pid, v, on in ADETAIL:
    R("  %-3s %-9s %-22s %s %s%s" % (qid, role, drug, pid,
      "   no ASC signature" if v is None else "%.4f" % v, "  ON THE TIED BLOCK" if on else ""))
R("--- ASC: the MEK1/2 class (declared secondary, reported, never gating) ---")
CLS = sorted(set(DR[DR.role == "class"].drug))
ACLS = {}
for qid in QALL:
    per = []
    for d in CLS:
        ids = sorted(DR[(DR.role == "class") & (DR.drug == d)].pert_id)
        vv = APCT[qid].reindex(ids).dropna().values
        if len(vv): per.append(float(np.median(vv)))
    ACLS[qid] = (float(np.median(per)), len(per)) if per else (None, 0)
    R("  ASC %s: median over %d of the %d class drugs with an ASC signature: %s"
      % (qid, ACLS[qid][1], len(CLS), "none present" if ACLS[qid][0] is None else round(ACLS[qid][0], 4)))
ACLS_STAT = None; ACLS_P = None
_live = [q for q in LIVE if ACLS[q][0] is not None]
if _live:
    ACLS_STAT = float(np.mean([ACLS[q][0] for q in _live]))
    sizes = []
    for d in CLS:
        ids = sorted(DR[(DR.role == "class") & (DR.drug == d)].pert_id)
        k = min(int(APCT[q].reindex(ids).notna().sum()) for q in _live)
        if k: sizes.append(k)
    commonC = sorted(set.intersection(*[set(APCT[q].index) for q in _live]))
    PC = {q: {p: i for i, p in enumerate(APCT[q].index)} for q in _live}
    cix = {q: np.array([PC[q][p] for p in commonC]) for q in _live}
    IX = {q: APCT[q].values for q in _live}
    rng2 = np.random.default_rng(777); CN = np.empty(2000); need = sum(sizes)
    assert need <= len(commonC), ("NOT ENOUGH ASC COMPOUNDS TO DRAW THE CLASS STRUCTURE", need, len(commonC))
    for b in range(2000):
        pick = rng2.choice(len(commonC), size=need, replace=False)
        tot = []
        for q in _live:
            i = 0; per = []
            for k in sizes:
                per.append(np.median(IX[q][cix[q][pick[i:i + k]]])); i += k
            tot.append(np.median(per))
        CN[b] = np.mean(tot)
    ACLS_P = (1 + int((CN <= ACLS_STAT).sum())) / 2001
    R("  ASC class statistic over %s: %.4f | null mean %.4f sd %.4f | p = %.4f (2,000 draws)"
      % (_live, ACLS_STAT, CN.mean(), CN.std(), ACLS_P))
R("--- ASC: reference compounds and the sensitivity query, reported only ---")
for qid in QALL:
    for d in sorted(set(DR[DR.role == "reference"].drug)):
        ids = sorted(DR[(DR.role == "reference") & (DR.drug == d)].pert_id)
        vv = APCT[qid].reindex(ids).dropna().values
        R("  ASC %-3s %-22s %s" % (qid, d, "no ASC signature" if not len(vv) else "%.4f" % float(np.median(vv))))
for q in SENS:
    R("  ASC %s: PD-0325901 %s | class %s | shares a laboratory with the P01 source data, so individuals"
      % (q, "no ASC signature" if APER[q] is None else "%.4f" % APER[q],
         "none present" if ACLS[q][0] is None else "%.4f" % ACLS[q][0]))
    R("     may overlap; it never enters the primary or this secondary's statistic.")

In [ ]:
# CELL 8: POST HOC. NOT PRE-REGISTERED. DESCRIPTIVE ONLY. NO P-VALUE IS COMPUTED FROM IT AND IT
# CANNOT CHANGE ANY CONCLUSION. It exists because B2 raised one question the pre-registration did not
# anticipate: PD-0325901's percentile in Q2 WAS the tied block, so Q2 contributed no compound-specific
# information, and S1 - the query that shares a laboratory with the P01 source - had both a lower
# exact-zero fraction and a far stronger MEK class signal. Those two facts have two different
# explanations: either the MEK signal is specific to that data source, or S1 simply resolves more
# compounds. Ranking only the compounds the instrument did NOT score as exactly zero separates them.
# A quantity computed after seeing the result is a description, not a test. It is labeled as such
# everywhere it appears and no inference is drawn from it in this notebook.
R("--- POST HOC, DESCRIPTIVE, NOT PRE-REGISTERED: rank among the compounds that scored nonzero ---")
NZ = {}
for qid in QALL:
    comp = COMP[qid]; nz = comp[comp.values != 0.0]
    if not len(nz): R("  %s: no compound scored nonzero" % qid); NZ[qid] = None; continue
    pct = pd.Series((midrank(nz.values) - 0.5) / len(nz), index=nz.index)
    got = [p for p in PD_IDS if p in pct.index]
    drug = float(np.median(pct.reindex(got).values)) if got else None
    cls = []
    for d in CLS:
        ids = sorted(DR[(DR.role == "class") & (DR.drug == d)].pert_id)
        vv = pct.reindex(ids).dropna().values
        if len(vv): cls.append(float(np.median(vv)))
    NZ[qid] = dict(n_nonzero=int(len(nz)), frac=float(len(nz) / len(comp)),
                   pd0325901=drug, n_pd_ids=len(got),
                   class_median=(float(np.median(cls)) if cls else None), n_class=len(cls))
    R("  %s: %d of %d compounds nonzero (%.1f%%) | PD-0325901 %s (%d of 2 ids nonzero) | MEK class %s (%d of %d drugs)"
      % (qid, len(nz), len(comp), 100 * len(nz) / len(comp),
         "not nonzero in this query" if drug is None else "%.4f" % drug, len(got),
         "none nonzero" if not cls else "%.4f" % float(np.median(cls)), len(cls), len(CLS)))
R("  Read this only as: of the compounds this instrument separated at all, where did they fall.")
R("  It is not a test of anything. The pre-registered criterion was settled in Phase B2 and is NOT MET.")

In [ ]:
# CELL 9: freeze the secondary result and print the report.
out = dict(prereg_sha256=PREREG, frozen_sha256=FREEZE, b2_result_sha256=B2SHA,
           live_queries=LIVE, asc_live=ALIVE, n_asc_signatures=int(ASC.sum()),
           n_asc_compounds={q: ANC[q] for q in QALL},
           asc_tie_frac=ATIE, asc_tie_block=AZERO, asc_per_query=APER,
           asc_stat=ASTAT, asc_p=AP, asc_class_stat=ACLS_STAT, asc_class_p=ACLS_P,
           full_tie_frac=TIE, asc_detail=ADETAIL, post_hoc_nonzero=NZ)
bts = json.dumps(out, sort_keys=True, default=lambda x: None if x is None else float(x) if isinstance(x, (np.floating, np.integer)) else str(x)).encode()
open("sscmek_B3_result.json", "wb").write(bts)
assert open("sscmek_B3_result.json", "rb").read() == bts, "RESULT READ-BACK MISMATCH"
np.save("sscmek_B3_asc_cells.npy", cells)
print("=== SSC-MEK PHASE B3 REPORT BEGIN ===")
print("\n".join(REPORT))
print("--- HEADLINE ---")
print("pre-registration", PREREG)
print("B2 result this is bound to", B2SHA)
print("THE PRE-REGISTERED PRIMARY WAS SETTLED IN PHASE B2: p = %.5f, CRITERION NOT MET." % B2["p"])
print("Phase B3 adds the declared ASC secondary only. It is reported and never gating.")
print("ASC signatures %d | ASC compounds per query %s" % (int(ASC.sum()), {q: ANC[q] for q in QALL}))
print("ASC exact-zero fraction %s   (full library %s)"
      % ({q: round(ATIE[q], 3) for q in QALL}, {q: round(TIE[q], 3) for q in QALL}))
for q in QALL:
    print("  ASC PD-0325901 in %s: %s" % (q, "no ASC signature" if APER[q] is None else "%.4f" % APER[q]))
print("ASC SECONDARY STATISTIC over %s: %s" % (ALIVE, "undefined" if ASTAT is None else "%.4f" % ASTAT))
print("ASC SECONDARY p = %s  [SECONDARY, NOT GATING]" % ("undefined" if AP is None else "%.5f" % AP))
print("ASC MEK1/2 class p = %s  [SECONDARY, NOT GATING]" % ("undefined" if ACLS_P is None else "%.4f" % ACLS_P))
print("sscmek_B3_result.json sha256", hashlib.sha256(bts).hexdigest())
print("=== SSC-MEK PHASE B3 REPORT END ===")